# Step 1 — Loading & Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/dataset-diabete.csv")

print(df.shape)
print(df.dtypes)
print(df.head())

print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

zero_check_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_check_cols:
    print(col, "zeros:", (df[col] == 0).sum())

print(df.describe())

In [ ]:
df.hist(figsize=(12, 10), bins=30)
plt.tight_layout()
plt.savefig("../notebooks/distributions.png")
plt.show()

# Step 2 — Preprocessing

## 2.1 Missing-value pattern (fake zeros treated as NaN)

In [ ]:
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df_check = df.copy()
df_check[cols] = df_check[cols].replace(0, np.nan)

missing_per_row = df_check[cols].isnull().sum(axis=1)
df_check["missing_count"] = missing_per_row

print(missing_per_row.value_counts().sort_index())

for n in [2, 3, 4]:
    subset = df_check[df_check["missing_count"] == n]
    print(f"\n=== Rows missing {n} values ({len(subset)} rows) ===")
    print(subset)

Findings: rows missing 2 values are always SkinThickness + Insulin together. Rows missing 3 add BloodPressure. Rows missing 4 (7 rows) are missing BloodPressure, SkinThickness, Insulin and BMI all at once — dropped below.

## 2.2 Imputation (MICE) + cleanup

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

df = pd.read_csv("../data/dataset-diabete.csv")

cols_with_fake_zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[cols_with_fake_zeros] = df[cols_with_fake_zeros].replace(0, np.nan)

missing_per_row = df[cols_with_fake_zeros].isnull().sum(axis=1)
df = df[missing_per_row < 4].reset_index(drop=True)

all_numeric_cols = df.columns.tolist()
imputer = IterativeImputer(random_state=42)
df[all_numeric_cols] = imputer.fit_transform(df[all_numeric_cols])

df["Pregnancies"] = df["Pregnancies"].round().astype(int)
df["Age"] = df["Age"].round().astype(int)

cols_min_zero = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction"]
df[cols_min_zero] = df[cols_min_zero].clip(lower=0)

print(df.isnull().sum())
print(df.shape)
print(df.describe())

## 2.3 Outlier detection (IQR)

Outliers kept as-is — flagged values are physiologically plausible, not data errors. Influence on clustering handled via RobustScaler below.

In [ ]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

numeric_cols = df.columns.tolist()

print(f"{'Column':<28} {'Lower':>10} {'Upper':>10} {'#Outliers':>10}")
for col in numeric_cols:
    low, high = iqr_bounds(df[col])
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col:<28} {low:>10.2f} {high:>10.2f} {n_out:>10}")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    ax.boxplot(df[col])
    ax.set_title(col)
plt.tight_layout()
plt.savefig("../notebooks/boxplots.png")
plt.show()

## 2.4 Correlation analysis + variable selection

Strongest correlations: SkinThickness-BMI (0.71), Glucose-Insulin (0.70). Keeping all 8 features — not high enough to count as redundant, both pairs clinically meaningful on their own.

In [ ]:
corr = df.corr()
print(corr)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../notebooks/correlation_heatmap.png")
plt.show()

In [ ]:
sns.pairplot(df, diag_kind="hist", corner=True)
plt.savefig("../notebooks/pairplot.png")
plt.show()

## 2.5 Scaling (RobustScaler)

RobustScaler used instead of StandardScaler so extreme Insulin values don't distort distance calculations in K-Means.

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
df_scaled = scaler.fit_transform(df)
df_scaled = pd.DataFrame(df_scaled, columns=df.columns)

print(df_scaled.describe())

# Step 3 — K-Means Clustering

## 3.1 Choosing k (elbow + silhouette)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 8)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(df_scaled, labels))
    print(f"k={k}: inertia={km.inertia_:.2f}, silhouette={silhouettes[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_range, inertias, marker="o")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(k_range, silhouettes, marker="o", color="orange")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Method")

plt.tight_layout()
plt.savefig("../notebooks/elbow_silhouette.png")
plt.show()

k=2 gives the highest silhouette score (0.2304), clearly ahead of k=3+.

## 3.2 Final model

In [ ]:
final_km = KMeans(n_clusters=2, random_state=42, n_init=10)
df["Cluster"] = final_km.fit_predict(df_scaled)

print(df["Cluster"].value_counts())
print(df.groupby("Cluster").mean())

In [ ]:
df["Cluster"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Cluster")
plt.ylabel("Number of patients")
plt.title("Cluster Distribution")
plt.savefig("../notebooks/cluster_distribution.png")
plt.show()

# Step 4 — Cluster Interpretation & Risk Labeling

Cluster 1 (296 patients) clears all three thresholds: Glucose 146.99 (>126), BMI 36.26 (>30), DPF 0.556 (>0.5) → high risk.
Cluster 0 (465 patients) does not → low risk.

In [ ]:
df["risk_category"] = df["Cluster"].map({0: 0, 1: 1})

print(df["risk_category"].value_counts())